In [1]:
import os
import glob
import pandas as pd


# get current working directory
path = os.getcwd()
  
# Extracting all the folders with csv data
folders = glob.glob(path + "/*-*")




In [2]:
# Data cleaning
def clean_data(df):
    # Drop rows with any missing values
    df.dropna(inplace=True)
    # Convert SEQN to int if it's n ot already
    if 'SEQN' in df.columns:
        df['SEQN'] = df['SEQN'].astype(int)

In [3]:

def process_data_from_files(files):
    for file in files:
            if(not file.endswith("merged_data.csv")):
                if(file.endswith(files[0]) ):
                    bmx_df  = pd.read_csv(file)
                    clean_data(bmx_df)                
                elif(file.endswith(files[1])):
                    data_frame = pd.read_csv(file)
                    clean_data(data_frame)
                    merged_data = pd.merge(bmx_df, data_frame, on='SEQN', how='inner')           
                else:
                    data_frame = pd.read_csv(file)
                    clean_data(data_frame)
                    merged_data = pd.merge(merged_data, data_frame, on='SEQN', how='inner')    
    return merged_data

In [4]:

def process_data_from_folders(folders):
    merged_data = pd.DataFrame()
    for folder in folders:
        files = glob.glob(folder + "/*.csv")
        merged_data = pd.concat([process_data_from_files(files=files),merged_data],axis=0)
    return merged_data

In [5]:
merged_data = process_data_from_folders(folders=folders)


In [6]:
merged_data.shape

(8290, 16)

In [7]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8290 entries, 0 to 415
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      8290 non-null   int32  
 1   BMXWT     8290 non-null   float64
 2   BMXBMI    8290 non-null   float64
 3   BMXWAIST  8290 non-null   float64
 4   BPQ020    8290 non-null   float64
 5   BPQ080    8290 non-null   float64
 6   BPQ090D   8290 non-null   float64
 7   BPXSY1    8290 non-null   float64
 8   BPXSY2    8290 non-null   float64
 9   BPXSY3    8290 non-null   float64
 10  RIAGENDR  8290 non-null   float64
 11  RIDAGEYR  8290 non-null   float64
 12  DIQ010    8290 non-null   float64
 13  LBDGLUSI  8290 non-null   float64
 14  MCQ300C   8290 non-null   float64
 15  LBDTCSI   8290 non-null   float64
dtypes: float64(15), int32(1)
memory usage: 1.0 MB


In [8]:
merged_data.head()

,SEQN,BMXWT,BMXBMI,BMXWAIST,BPQ020,BPQ080,BPQ090D,BPXSY1,BPXSY2,BPXSY3,RIAGENDR,RIDAGEYR,DIQ010,LBDGLUSI,MCQ300C,LBDTCSI
0,93711,62.1,21.3,86.6,2.0,1.0,1.0,108.0,94.0,102.0,1.0,56.0,2.0,5.94,2.0,6.15
1,93718,54.4,22.0,77.5,1.0,2.0,2.0,128.0,136.0,130.0,1.0,45.0,2.0,4.94,1.0,3.93
2,93721,85.1,35.9,113.2,2.0,2.0,2.0,132.0,136.0,140.0,2.0,60.0,2.0,5.77,2.0,3.15
3,93722,56.8,23.8,89.7,2.0,2.0,2.0,116.0,110.0,112.0,2.0,60.0,2.0,5.61,2.0,4.76
4,93735,76.5,35.1,104.6,1.0,1.0,1.0,154.0,156.0,146.0,1.0,52.0,2.0,4.61,2.0,3.13


In [9]:
# Calculate average of 'BPXSY1', 'BPXSY2', and 'BPXSY3' columns
merged_data['BPXSY_avg'] = merged_data[['BPXSY1', 'BPXSY2', 'BPXSY3']].mean(axis=1)

# Remove 'BPXSY1', 'BPXSY2', and 'BPXSY3' columns
merged_data.drop(['BPXSY1', 'BPXSY2', 'BPXSY3'], axis=1, inplace=True)

In [10]:
# Columns to convert to int64
columns_to_convert = ['BPQ020', 'BPQ080', 'BPQ090D', 'RIAGENDR', 'RIDAGEYR', 'DIQ010', 'MCQ300C']

# Convert columns to int64
merged_data[columns_to_convert] = merged_data[columns_to_convert].astype('int64')

In [11]:
# BPQ020,BPQ080,BPQ090D,DIQ010,MCQ300C: 1=yes; 2=no --change to--> 1=yes; 0=no

# Columns to replace values in
columns_to_replace = ['BPQ020', 'BPQ080', 'BPQ090D', 'DIQ010', 'MCQ300C']

# Replace all occurrences of 2 with 0 in specified columns
merged_data[columns_to_replace] = merged_data[columns_to_replace].replace(2, 0)

In [12]:
# RIAGENDR: 1=Male; 2=Female --change to--> 0=Male; 1=Female

# Replace values in the 'RIAGENDR' column
merged_data['RIAGENDR'] = merged_data['RIAGENDR'].replace({1: 0, 2: 1})

In [13]:
# Save the updated merged dataframe to a new CSV file
merged_data.to_csv("merged_data.csv", index=False)